# Exploring the features of the `SparseLinearCustom` class 

> `SparseLinearCustom` requires more inputs than a linear layer. This notebook explores why. 

In [ ]:
import torch
from sparsevnn.core import *

With this class customized weights can now be passed into the layer. This is _great_ because now the problem of defining the structure of the matrices outside of this function and pass the values in. This gives us two compleicateted functions rather than one *behemouth*. 

In [ ]:
# #
# # 
    #
      #

SparseLinearCustom(
    4, 4,
    connectivity   = torch.LongTensor(torch.tensor([[0, 0, 1, 1, 2, 3],
                                                    [0, 1, 0, 1, 2, 3]])),
    custom_weights = torch.tensor([-0.2665,  0.3926, -0.2531,  0.3266,  1.0000, 1.0000]), 
    custom_bias    = torch.tensor([-0.2665,  0.3926, -0.2531,  0.3266,  1.0000, 1.0000])
).weight.to_dense()

That's not all. I've extened this class in two other important ways. 

First, I've added attributes for which values (if any) in the weight and biases gradients should be zeroed. Why? If we want to reprent an identity function (say nodes placed in two layers require the same input) then the weight and bias should always be 1 and 0. There's no way to freeze part of a matrix so we'll have to zero the gradient on these values before stepping the optimizer. 

Second, I've added dropout. This seems like extra work - shouldn't a sparse linear layer's output just go into `nn.Dropout`? That would be simpler, but we want to _exempt specific parts of the matrix_ from dropout. Just like we didn't want to update the gradients on idenity matrices we don't want the outputs from these to be zeroed. If dropout is used they will have a chance of being zeroed _when they're first calculated_. 


For intuition on the consequences of this, condider the following graph. Each letter represents a layer in the network. 
```t
A -- B -- C -- D
 \____________/
```
Importantly layer D take the output of layer A and C as inputs. If each of these intermediate outputs were stored separately then we would have no problem using `nn.Dropout`, but we're not storing them separately. We'll end up with sparse matrices with outputs for the layers (A), (A,B), (A,C), (D). If the values from A are not protected from dropout then the number of zeroed values will increase. For a dropout rate of 0.5, B will receive ~50% intact values but D will only receive ~12%! (.5**3)

Let's test this using a case that will make it easy to see if the system is misbehaving. The weights will be an identity matrix and bias will be 0 so the output should mirror the input. We'll pass in a matix of "2"s. 

In [ ]:
eye4 = torch.eye(4).to_sparse()           # Identity matrix
eye4_c = torch.LongTensor(eye4.indices()) # Connectivity in sparse format
eye4_w = eye4.values()                    # Values in sparse format
eye4_b = torch.tensor([0., 0, 0, 0])      # Bias vector

x = (torch.ones(4)+1).repeat(5, 1)        # "Data"


Success! The layer returns `x`. 

In [ ]:
SparseLinearCustom(
    4, 4, connectivity=eye4_c, custom_weights=eye4_w, custom_bias=eye4_b
    )(x)

Now we'll set a high dropout rate to see the effect.

In [ ]:
SparseLinearCustom(
    4, 4, connectivity=eye4_c, custom_weights=eye4_w, custom_bias=eye4_b,
    dropout_p = 0.8
    )(x)

Looks good. Now let's only drop values in half the columns. A 1 here means the output shouldn't be zeroed and a 2 means that it can. These values are used instead of 0 and 1 to make sure the number values does not change for a sparse representation (where only the non-zero values would be stored). This saves us some headaches elsehere so we'll be consistent with that convention here. .

In [ ]:
SparseLinearCustom(
    4, 4, connectivity=eye4_c, custom_weights=eye4_w, custom_bias=eye4_b,
    bias_grad_bool = torch.tensor([1., 1, 2, 2]),
    dropout_p = 0.8 
    )(x)

It seems to have worked but there's a problem here. We only want to drop values while the model is fitting. We want _all_ of the outputs when it's time to make inferences. Pytorch uses the `training` flag to track which mode the model is in. We can check this by calling `SparseLinearCustom(...).training` which will return `True`. We can change this flag by using the `eval()` and `train()` methods. To confirm this works, we'll take the same model, set it to evaluation mode and _no_ values should be dropped.

In [ ]:
SparseLinearCustom(
    4, 4, connectivity=eye4_c, custom_weights=eye4_w, custom_bias=eye4_b,
    bias_grad_bool = torch.tensor([1., 1, 2, 2]),
    dropout_p = 0.8
    ).eval()(x)

We're almost done but there's two more tricks left. What if different nodes need different dropout percentages? As is we can only apply a single dropout to all the non-identitiy outputs.

To get around this, let's use a tensor of probabilites. 

In [ ]:
SparseLinearCustom(
    4, 4, connectivity=eye4_c, custom_weights=eye4_w, custom_bias=eye4_b,
    dropout_p = torch.tensor([0.25, .5, .75, 1.])
    )(x)

What if we make a mistake and allow dropout for an output that shouldn't be? Not to worry. `bias_grad_bool` will set the dropout probability of these cells to 0 before it's applied. 

In [ ]:
SparseLinearCustom(
    4, 4, connectivity=eye4_c, custom_weights=eye4_w, custom_bias=eye4_b,
    bias_grad_bool = torch.tensor([1., 1, 2, 2]),
    dropout_p = torch.tensor([0.25, .5, .75, 1.])
    )(x)

Last trick. What happens when we pass the output of this layer into a ReLU or other transformation? _All_ the outputs are transformed. This means that if we want to preserve identity outputs then we'll need to run the non-linear transform within `SparseLinearCustom` and return the raw values for the identity cells. We can use the same trick used for dropout (condition the output on the `bias_gradient_bool` tensor). To show this in action let's change the weights in the layer from ones to -2 to +1. This will give us the following:

In [ ]:
SparseLinearCustom(
    4, 4, connectivity=eye4_c, custom_weights=torch.tensor([-2., -1, 0, 1]), custom_bias=eye4_b,
    )(x)

Now we add in our non-linearity. Here I'm using `F.relu` so the first two columns should be zeroed. 

In [ ]:
import torch.nn.functional as F

In [ ]:
SparseLinearCustom(
    4, 4, connectivity=eye4_c, custom_weights=torch.tensor([-2., -1, 0, 1]), custom_bias=eye4_b,
    nonlinear_transform = F.relu
    )(x)

This is equivalent to `nn.ReLU( SparseLinearCustom(...)(x) )` to see if this is behaving correctly we'll specify that the first two columns should not recieve a gradient. Now those negative values should come back.  

In [ ]:
SparseLinearCustom(
    4, 4, connectivity=eye4_c, custom_weights=torch.tensor([-2., -1, 0, 1]), custom_bias=eye4_b,
    bias_grad_bool = torch.tensor([1., 1, 2, 2]),
    nonlinear_transform = F.relu
    )(x)